[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/notebooks/ch06_td_learning.ipynb)

<div style="background: linear-gradient(135deg, #1a1000 0%, #3a2800 50%, #6a4800 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #f0a500; font-size: 2.2em; margin: 0 0 10px 0;">Chapter 6: Temporal-Difference Learning</h1>
  <p style="color: #a8dadc; font-size: 1.1em; margin: 0;">TD(0), SARSA, Q-Learning, n-step TD, and TD(λ) with eligibility traces on CliffWalking-v0.</p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Environment Setup</strong><br>
  <span style="color: #cdd9e5;">Requires <code>gymnasium</code>. Estimated runtime: ~2 minutes on Colab free CPU.</span>
</div>

In [ ]:
!pip install gymnasium numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import gymnasium as gym

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})

env = gym.make('CliffWalking-v0')
print(f'CliffWalking-v0 loaded.')
print(f'States: {env.observation_space.n}, Actions: {env.action_space.n}')
print(f'Action map: 0=Up, 1=Right, 2=Down, 3=Left')
print(f'Grid: 4 rows × 12 cols. Start: (3,0). Goal: (3,11). Cliff: (3,1)-(3,10).')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500;">
  <strong style="color: #f0a500;">1. The CliffWalking Environment</strong>
</div>

**CliffWalking-v0** (Sutton & Barto Example 6.6): a 4×12 grid where the bottom row is a cliff.
- **Start**: bottom-left (state 36), **Goal**: bottom-right (state 47)
- **Cliff**: states 37–46 — stepping on the cliff gives reward $-100$ and resets to start.
- All other steps: reward $-1$.

This environment perfectly illustrates the **SARSA vs Q-Learning** difference: SARSA learns a safe path hugging the top, Q-Learning learns the optimal (but risky) cliff-edge path.

In [ ]:
# ── Visualise the CliffWalking environment ──
GRID_ROWS, GRID_COLS = 4, 12

fig, ax = plt.subplots(figsize=(12, 4))
for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        s = r * GRID_COLS + c
        if r == 3 and 1 <= c <= 10:   # Cliff
            color = '#da3633'
            label = 'CLIFF\n-100'
        elif r == 3 and c == 0:        # Start
            color = '#1f6feb'
            label = 'S'
        elif r == 3 and c == 11:       # Goal
            color = '#2ea043'
            label = 'G'
        else:
            color = '#21262d'
            label = str(s)
        rect = mpatches.FancyBboxPatch(
            (c + 0.05, (3-r) + 0.05), 0.9, 0.9,
            boxstyle='round,pad=0.04', linewidth=1,
            edgecolor='#30363d', facecolor=color
        )
        ax.add_patch(rect)
        ax.text(c + 0.5, (3-r) + 0.5, label, ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')

ax.set_xlim(0, GRID_COLS)
ax.set_ylim(0, GRID_ROWS)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('CliffWalking-v0 (4×12 Grid)', fontsize=13)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500;">
  <strong style="color: #f0a500;">2. TD(0) Prediction, SARSA, and Q-Learning</strong>
</div>

**TD(0)** prediction update:
$$V(S_t) \leftarrow V(S_t) + \alpha\left[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\right]$$

**SARSA** (on-policy): uses the next action $A_{t+1}$ already chosen by the policy:
$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)\right]$$

**Q-Learning** (off-policy): bootstraps with the greedy next action:
$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma \max_{a'} Q(S_{t+1},a') - Q(S_t,A_t)\right]$$

In [ ]:
# ── SARSA Implementation ──

def sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1, seed=None):
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q = np.zeros((n_s, n_a))
    episode_rewards = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed)
        # Choose A from S using ε-greedy
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[obs]))

        total_r = 0
        done = False
        while not done:
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_r += reward

            # Choose A' using ε-greedy
            if np.random.rand() < epsilon:
                next_action = env.action_space.sample()
            else:
                next_action = int(np.argmax(Q[next_obs]))

            # SARSA update
            td_target = reward + gamma * Q[next_obs, next_action] * (not done)
            Q[obs, action] += alpha * (td_target - Q[obs, action])

            obs, action = next_obs, next_action

        episode_rewards.append(total_r)

    return Q, episode_rewards


# ── Q-Learning Implementation ──

def q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1, seed=None):
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q = np.zeros((n_s, n_a))
    episode_rewards = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed)
        total_r = 0
        done = False
        while not done:
            # ε-greedy action selection
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[obs]))

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_r += reward

            # Q-Learning update: greedy max
            td_target = reward + gamma * np.max(Q[next_obs]) * (not done)
            Q[obs, action] += alpha * (td_target - Q[obs, action])
            obs = next_obs

        episode_rewards.append(total_r)

    return Q, episode_rewards


print('Running SARSA (500 episodes)...')
Q_sarsa, rewards_sarsa = sarsa(env, n_episodes=500, alpha=0.5, epsilon=0.1, seed=SEED)

print('Running Q-Learning (500 episodes)...')
Q_ql, rewards_ql = q_learning(env, n_episodes=500, alpha=0.5, epsilon=0.1, seed=SEED)

print(f'SARSA  — Avg reward (last 50): {np.mean(rewards_sarsa[-50:]):.1f}')
print(f'Q-Lrn  — Avg reward (last 50): {np.mean(rewards_ql[-50:]):.1f}')

In [ ]:
# ── Learning curves comparison ──
def smooth(arr, w=20):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smooth(rewards_sarsa), color='#58a6ff', linewidth=2, label='SARSA (on-policy)')
ax.plot(smooth(rewards_ql),    color='#ffd700', linewidth=2, label='Q-Learning (off-policy)')
ax.axhline(-13, color='#56d364', linestyle='--', alpha=0.7, label='Optimal path reward (-13)')
ax.set_ylim(-200, 0)
ax.set_title('CliffWalking: SARSA vs Q-Learning Learning Curves', fontsize=13)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Episode Reward (20-ep avg)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500;">
  <strong style="color: #f0a500;">3. Visualising the Paths: SARSA vs Q-Learning</strong>
</div>

With ε-greedy during training, SARSA avoids the cliff (risky path) because any accidental exploration step leads to a large penalty — the policy learns to stay safe. Q-Learning learns the theoretically optimal path along the cliff edge, but this is unstable during training.

In [ ]:
# ── Extract greedy paths from learned Q-tables ──

def extract_path(Q, env, max_steps=100, seed=0):
    obs, _ = env.reset(seed=seed)
    path = [obs]
    done = False
    for _ in range(max_steps):
        action = int(np.argmax(Q[obs]))   # Greedy (no exploration)
        obs, _, terminated, truncated, _ = env.step(action)
        path.append(obs)
        done = terminated or truncated
        if done:
            break
    return path


def state_to_rc(s, cols=12):
    return divmod(s, cols)


path_sarsa = extract_path(Q_sarsa, env)
path_ql    = extract_path(Q_ql, env)

print(f'SARSA path length:    {len(path_sarsa)} steps')
print(f'Q-Learning path length: {len(path_ql)} steps')


# ── Plot paths on grid ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (path, algo, color) in zip(
    axes,
    [(path_sarsa, 'SARSA (safe path)', '#58a6ff'),
     (path_ql,    'Q-Learning (cliff edge)', '#ffd700')]
):
    # Draw grid
    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            if r == 3 and 1 <= c <= 10:
                fc = '#da3633'
            elif r == 3 and c == 0:
                fc = '#1f6feb'
            elif r == 3 and c == 11:
                fc = '#2ea043'
            else:
                fc = '#21262d'
            rect = mpatches.FancyBboxPatch(
                (c + 0.05, (3-r) + 0.05), 0.9, 0.9,
                boxstyle='round,pad=0.04', linewidth=1,
                edgecolor='#30363d', facecolor=fc
            )
            ax.add_patch(rect)

    # Draw path
    for i in range(len(path) - 1):
        r1, c1 = state_to_rc(path[i])
        r2, c2 = state_to_rc(path[i+1])
        ax.annotate('',
                    xy=(c2 + 0.5, (3-r2) + 0.5),
                    xytext=(c1 + 0.5, (3-r1) + 0.5),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.2))

    ax.set_xlim(0, GRID_COLS)
    ax.set_ylim(0, GRID_ROWS)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{algo}  ({len(path)-1} steps)', fontsize=11)

plt.suptitle('Greedy Policy Paths after Training', fontsize=13)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500;">
  <strong style="color: #f0a500;">4. n-Step TD &amp; TD(λ) with Eligibility Traces</strong>
</div>

**n-step TD** generalises between TD(0) and MC by using $n$-step returns:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

**TD(λ)** uses eligibility traces to blend all $n$-step returns at once:
$$e_t(s) = \gamma\lambda\, e_{t-1}(s) + \mathbb{1}[S_t = s]$$
$$V(s) \leftarrow V(s) + \alpha\,\delta_t\, e_t(s) \quad \text{for all } s$$

where $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is the TD error.

In [ ]:
# ── SARSA(λ) with Replacing Eligibility Traces ──

def sarsa_lambda(env, n_episodes=500, alpha=0.5, gamma=1.0,
                 epsilon=0.1, lam=0.9, seed=None):
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q = np.zeros((n_s, n_a))
    episode_rewards = []

    for ep in range(n_episodes):
        E = np.zeros((n_s, n_a))   # Eligibility traces
        obs, _ = env.reset(seed=seed)

        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[obs]))

        total_r = 0
        done = False
        while not done:
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_r += reward

            if np.random.rand() < epsilon:
                next_action = env.action_space.sample()
            else:
                next_action = int(np.argmax(Q[next_obs]))

            delta = reward + gamma * Q[next_obs, next_action] * (not done) - Q[obs, action]

            # Replacing traces
            E[obs, action] = 1.0

            # Update all (s,a) with nonzero trace
            Q += alpha * delta * E
            E *= gamma * lam

            obs, action = next_obs, next_action

        episode_rewards.append(total_r)

    return Q, episode_rewards


# Compare SARSA, Q-Learning, SARSA(λ)
lambdas = [0.0, 0.5, 0.9]
results_lambda = {}

print('Running SARSA(λ) variants...')
for lam in lambdas:
    np.random.seed(SEED)
    _, rews = sarsa_lambda(env, n_episodes=500, alpha=0.4, lam=lam, seed=SEED)
    results_lambda[f'SARSA(λ={lam})'] = rews
    print(f'  λ={lam}  avg last-50 reward: {np.mean(rews[-50:]):.1f}')

In [ ]:
# ── Final comparison plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot SARSA vs Q-Learning (full 500 episodes)
ax = axes[0]
ax.plot(smooth(rewards_sarsa, 20), color='#58a6ff', lw=2, label='SARSA')
ax.plot(smooth(rewards_ql, 20),    color='#ffd700', lw=2, label='Q-Learning')
ax.axhline(-13, color='#56d364', linestyle='--', alpha=0.6, label='Optimal (-13)')
ax.set_ylim(-250, 5)
ax.set_title('SARSA vs Q-Learning', fontsize=12)
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward (20-ep avg)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot SARSA(λ) for different lambdas
ax = axes[1]
cmap = plt.cm.plasma
for i, (name, rews) in enumerate(results_lambda.items()):
    color = cmap(i / len(results_lambda))
    ax.plot(smooth(rews, 20), color=color, lw=2, label=name)
ax.axhline(-13, color='#56d364', linestyle='--', alpha=0.6, label='Optimal (-13)')
ax.set_ylim(-250, 5)
ax.set_title('SARSA(λ): Effect of λ on Learning', fontsize=12)
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward (20-ep avg)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('TD Learning on CliffWalking-v0', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500;">
  <strong style="color: #f0a500;">5. Algorithm Comparison Summary</strong>
</div>

In [ ]:
# ── Summary table ──
all_algos = {
    'SARSA':          rewards_sarsa,
    'Q-Learning':     rewards_ql,
    **results_lambda,
}

print(f"{'Algorithm':<20} {'Avg Reward (last 50)':<25} {'Best Episode'}")
print('-' * 60)
for name, rews in all_algos.items():
    avg = np.mean(rews[-50:])
    best = max(rews)
    print(f'{name:<20} {avg:<25.2f} {best:.1f}')


# ── Final bar chart ──
fig, ax = plt.subplots(figsize=(10, 4))
names = list(all_algos.keys())
avgs  = [np.mean(all_algos[n][-50:]) for n in names]
colors = ['#58a6ff', '#ffd700', '#f0a500', '#e94560', '#c792ea']

bars = ax.barh(names, avgs, color=colors[:len(names)], alpha=0.85)
ax.axvline(-13, color='#56d364', linestyle='--', alpha=0.8, label='Optimal (-13)')
for bar, v in zip(bars, avgs):
    ax.text(v + 2, bar.get_y() + bar.get_height()/2, f'{v:.1f}',
            va='center', fontsize=10, color='white')
ax.set_xlabel('Average Episode Reward (last 50 episodes)')
ax.set_title('Algorithm Comparison: CliffWalking-v0', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Chapter 6 — Key Takeaways</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li><strong>TD(0)</strong> bootstraps from the next state — combines the sampling of MC with the bootstrapping of DP.</li>
    <li><strong>SARSA</strong> (on-policy) is safer in stochastic environments — it accounts for exploration in its updates.</li>
    <li><strong>Q-Learning</strong> (off-policy) converges to the optimal policy, even if the behaviour policy is exploratory.</li>
    <li>On CliffWalking, SARSA learns a <em>safe</em> path away from the cliff; Q-Learning learns the <em>optimal</em> cliff-edge path.</li>
    <li><strong>TD(λ)</strong> with eligibility traces smoothly bridges TD(0) (λ=0) and MC (λ=1), often learning faster than either extreme.</li>
  </ul>
</div>